# 3. Pace components

### Purpose

Calculate pace from the event transitions and shared modeled timeline exported by [notebook 02](02_data_augmentation.ipynb). Use [Playing Fast Not Loose](../media/Playing%20Fast%20Not%20Loose.pdf) as the methodological reference.

### Expected outcome

Transition-level and aggregated pace tables. Recorded clocks remain available; speeds use the estimated within-second timing from section 2.4 of notebook 02.

## 3.1. Setup and load augmented data

Load `events_augmented.parquet` and `event_transitions.parquet` exported in notebook 02 section 2.7, plus teams and games. Transitions include recorded and modeled endpoint clocks.

In [49]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)

ROOT_DIR = Path("..")
PROCESSED_DATA_DIR = ROOT_DIR / "data" / "processed"

augmented_events_df = pd.read_parquet(PROCESSED_DATA_DIR / "events_augmented.parquet")
transitions_df = pd.read_parquet(PROCESSED_DATA_DIR / "event_transitions.parquet")
teams_df = pd.read_parquet(PROCESSED_DATA_DIR / "teams.parquet")
games_df = pd.read_parquet(PROCESSED_DATA_DIR / "games.parquet")

# Calculate pace on a copy of the augmented transitions.
pace_df = transitions_df.copy()

loaded_tables = {
    "augmented_events": augmented_events_df,
    "transitions": transitions_df,
    "teams": teams_df,
    "games": games_df,
}
display(pd.DataFrame([
    {"table": name, "rows": len(df), "columns": len(df.columns)}
    for name, df in loaded_tables.items()
]))


,table,rows,columns
0,augmented_events,86670,27
1,transitions,63298,15
2,teams,20,3
3,games,34,5


## 3.2. Define eligible pace intervals

### Recorded elapsed time

Subtract the end clock from the start clock. Preserve recorded timestamps.

In [50]:
# Recorded clocks count down within each period.
pace_df["elapsed_seconds"] = (
    pace_df["start_clock_seconds"] - pace_df["end_clock_seconds"]
)

### Modeled elapsed time

Subtract exported modeled endpoint clocks. Notebook 02 assigns one game-wide timeline across possession changes, sharing times for consecutive identical positions from the same team. Notebook 03 does not redistribute timestamps. Movement links remain within possession sequences.

In [51]:
# Consume modeled clocks from notebook 02; do not recompute event timing.
pace_df["modeled_elapsed_seconds"] = (
    pace_df["start_modeled_clock_seconds"] - pace_df["end_modeled_clock_seconds"]
)


### Sequence endings and exclusions

Keep final movements. Exclude invalid recorded or modeled times and coordinates. Zero-modeled-duration movements remain in distance totals; individual speeds are missing. The 0:00 boundary can remain tied.

In [52]:
# Keep both recorded and modeled timing; only invalid intervals are excluded.
pace_df["exclusion_reason"] = pd.Series(pd.NA, index=pace_df.index, dtype="string")
coordinates = pace_df[["start_x", "start_y", "end_x", "end_y"]]
clocks = pace_df[["start_clock_seconds", "end_clock_seconds"]]
modeled_clocks = pace_df[["start_modeled_clock_seconds", "end_modeled_clock_seconds"]]
valid_position = (
    pace_df["start_x"].between(0, 200)
    & pace_df["end_x"].between(0, 200)
    & pace_df["start_y"].between(0, 85)
    & pace_df["end_y"].between(0, 85)
).fillna(False)
valid_clock = (np.isfinite(clocks) & clocks.ge(0)).all(axis=1)
valid_modeled_clock = (np.isfinite(modeled_clocks) & modeled_clocks.ge(0)).all(axis=1)

for reason, mask in [
    ("missing_clock", clocks.isna().any(axis=1)),
    ("invalid_clock", ~valid_clock),
    ("missing_position", coordinates.isna().any(axis=1)),
    ("invalid_position", ~valid_position),
    ("negative_elapsed_time", pace_df["elapsed_seconds"].lt(0)),
    ("missing_modeled_time", modeled_clocks.isna().any(axis=1)),
    ("invalid_modeled_clock", ~valid_modeled_clock),
    ("negative_modeled_time", pace_df["modeled_elapsed_seconds"].lt(0)),
]:
    pace_df.loc[
        pace_df["exclusion_reason"].isna() & mask.fillna(False),
        "exclusion_reason",
    ] = reason

assert len(pace_df) == len(transitions_df)
pd.testing.assert_frame_equal(pace_df[transitions_df.columns], transitions_df)
eligible = pace_df["exclusion_reason"].isna()
assert pace_df.loc[eligible, "elapsed_seconds"].ge(0).all()
assert pace_df.loc[eligible, "modeled_elapsed_seconds"].ge(0).all()
assert valid_position.loc[eligible].all()
assert valid_clock.loc[eligible].all()
assert valid_modeled_clock.loc[eligible].all()

# Compare recorded and exported modeled zero-duration counts.
display(pace_df.assign(
    recorded_zero=pace_df["elapsed_seconds"].eq(0),
    modeled_zero=pace_df["modeled_elapsed_seconds"].eq(0),
    modeled_positive=eligible & pace_df["modeled_elapsed_seconds"].gt(0),
    excluded=~eligible,
).groupby("source_dataset").agg(
    transitions=("end_event_id", "size"), recorded_zero=("recorded_zero", "sum"),
    modeled_zero=("modeled_zero", "sum"), modeled_positive=("modeled_positive", "sum"),
    excluded=("excluded", "sum"),
))

negative_time = pace_df["elapsed_seconds"].lt(0).fillna(False)
if negative_time.any():
    display(pace_df.loc[negative_time])
    raise ValueError("Negative elapsed time requires review before calculating pace.")


,transitions,recorded_zero,modeled_zero,modeled_positive,excluded
source_dataset,,,,,
NWHL,26777,11711,3778,22999,0
Olympics 2022,11258,5136,1108,10150,0
Womens,25263,11394,3313,21950,0


## 3.3. Calculate distance and speed components

### Displacement and distance

Calculate signed x/y displacement and straight-line distance in rink feet. Increasing x is forward for the sequence owner. Keep same-second distances.

In [53]:
# Straight-line distances between recorded positions, in rink feet.
# Distance is independent of recorded or modeled timing.
pace_df["dx_ft"] = (pace_df["end_x"] - pace_df["start_x"]).where(valid_position)
pace_df["dy_ft"] = (pace_df["end_y"] - pace_df["start_y"]).where(valid_position)
pace_df["distance_ft"] = np.hypot(pace_df["dx_ft"], pace_df["dy_ft"])


### Directional speeds

Calculate total, east-west, north-south, and north-only speed in feet per second using positive modeled durations. Backward movement has zero north-only speed. Retain recorded timing for comparison.

In [54]:
# Calculate individual speeds from positive modeled durations.
# Identical-position ties retain zero duration and missing individual speed.
can_calculate_speed = (
    pace_df["exclusion_reason"].isna()
    & pace_df["modeled_elapsed_seconds"].gt(0)
)
duration = pace_df["modeled_elapsed_seconds"].where(can_calculate_speed)
pace_df["speed_total_ft_s"] = pace_df["distance_ft"] / duration
pace_df["speed_ew_ft_s"] = pace_df["dy_ft"].abs() / duration
pace_df["speed_ns_ft_s"] = pace_df["dx_ft"].abs() / duration
pace_df["speed_n_ft_s"] = pace_df["dx_ft"].clip(lower=0) / duration

speed_columns = [
    "speed_total_ft_s", "speed_ew_ft_s", "speed_ns_ft_s", "speed_n_ft_s",
]
assert pace_df.loc[~can_calculate_speed, speed_columns].isna().all().all()
assert np.isfinite(pace_df.loc[can_calculate_speed, speed_columns]).all().all()
assert pace_df.loc[can_calculate_speed, speed_columns].ge(0).all().all()
assert np.allclose(
    pace_df.loc[can_calculate_speed, "speed_total_ft_s"].to_numpy(dtype=float),
    np.hypot(
        pace_df.loc[can_calculate_speed, "speed_ew_ft_s"].to_numpy(dtype=float),
        pace_df.loc[can_calculate_speed, "speed_ns_ft_s"].to_numpy(dtype=float),
    ),
)
assert pace_df.loc[can_calculate_speed, "speed_n_ft_s"].le(
    pace_df.loc[can_calculate_speed, "speed_ns_ft_s"]
).all()

print("Speeds use estimated within-second timing; recorded elapsed time is retained.")
display(pace_df[[
    "start_event_id", "end_event_id", "elapsed_seconds", "modeled_elapsed_seconds", "dx_ft", "dy_ft",
    "distance_ft", *speed_columns,
]].head(10))


Speeds use estimated within-second timing; recorded elapsed time is retained.


,start_event_id,end_event_id,elapsed_seconds,modeled_elapsed_seconds,dx_ft,dy_ft,distance_ft,speed_total_ft_s,speed_ew_ft_s,speed_ns_ft_s,speed_n_ft_s
0,1,2,2.0,2.0,7.0,-3.0,7.615773,3.807887,1.5,3.5,3.5
1,2,3,1.0,1.0,18.0,-12.0,21.633308,21.633308,12.0,18.0,18.0
2,3,4,2.0,2.0,6.0,0.0,6.0,3.0,0.0,3.0,3.0
3,5,6,1.0,1.0,-10.0,5.0,11.18034,11.18034,5.0,10.0,0.0
4,7,8,2.0,2.0,24.0,-25.0,34.655447,17.327723,12.5,12.0,12.0
5,8,9,1.0,1.0,32.0,-6.0,32.557641,32.557641,6.0,32.0,32.0
6,9,10,0.0,0.5,41.0,38.0,55.901699,111.803399,76.0,82.0,82.0
7,10,11,4.0,3.5,-5.0,-18.0,18.681542,5.337583,5.142857,1.428571,0.0
8,11,12,0.0,0.5,3.0,13.0,13.341664,26.683328,26.0,6.0,6.0
9,13,14,0.0,0.0,0.0,0.0,0.0,<NA>,<NA>,<NA>,<NA>


## 3.4. Calculate zonal components

### Zone transitions

Assign each transition to its destination zone, including its full distance and time when crossing a blue line. Segment identifiers are scoped to each possession. Shared boundary events connect adjacent intervals without duplicating distance or time.

In [55]:
# Keep aggregation inputs separate from the transition-level pace table.
aggregation_df = pace_df.copy()
aggregation_df["distance_ew_ft"] = aggregation_df["dy_ft"].abs()
aggregation_df["distance_ns_ft"] = aggregation_df["dx_ft"].abs()
aggregation_df["distance_n_ft"] = aggregation_df["dx_ft"].clip(lower=0)

# Zone labels use the possession team's increasing-x attacking direction.
def zone_from_x(x):
    zone = pd.Series(pd.NA, index=x.index, dtype="string")
    zone.loc[x.between(0, 75, inclusive="left")] = "DZ"
    zone.loc[x.between(75, 125)] = "NZ"
    zone.loc[x.gt(125) & x.le(200)] = "OZ"
    return zone

zonal_transitions_df = aggregation_df.sort_values("end_event_id", kind="stable").copy()
zonal_transitions_df["start_zone"] = zone_from_x(zonal_transitions_df["start_x"])
zonal_transitions_df["zone"] = zone_from_x(zonal_transitions_df["end_x"])
# Each interval belongs to its destination zone. A crossing interval
# enters the next segment; its end event anchors the following interval.
previous_zone = zonal_transitions_df.groupby("possession_id", sort=False)["zone"].shift()
new_segment = zonal_transitions_df["zone"].ne(previous_zone).fillna(True)
zonal_transitions_df["zone_segment_id"] = (
    new_segment.groupby(zonal_transitions_df["possession_id"], sort=False)
    .cumsum().astype("Int64")
)
zonal_transitions_df.loc[
    zonal_transitions_df["exclusion_reason"].isna()
    & (zonal_transitions_df["start_zone"].isna() | zonal_transitions_df["zone"].isna()),
    "exclusion_reason",
] = "missing_zone"


### Zonal pace

Sum component distances and modeled time by zone segment. Preserve recorded time totals. Leave speeds missing for zero-modeled-duration or incomplete segments.

In [56]:
distance_columns = ["distance_ft", "distance_ew_ft", "distance_ns_ft", "distance_n_ft"]
component_columns = dict(zip(distance_columns, [
    "speed_total_ft_s", "speed_ew_ft_s", "speed_ns_ft_s", "speed_n_ft_s",
]))

# Pool distances and modeled time; preserve recorded totals for comparison.
def summarize_pace(intervals, keys):
    work = intervals.copy()
    valid = work["exclusion_reason"].isna()
    work["excluded_transitions"] = (~valid).astype("Int64")
    work["same_second_transitions"] = (valid & work["elapsed_seconds"].eq(0)).astype("Int64")
    for column in ["elapsed_seconds", "modeled_elapsed_seconds", *distance_columns]:
        work[column] = work[column].where(valid)
    totals = work.groupby(keys, dropna=False, sort=False).agg(
        transitions=("end_event_id", "size"),
        sequences=("possession_id", "nunique"),
        excluded_transitions=("excluded_transitions", "sum"),
        same_second_transitions=("same_second_transitions", "sum"),
        elapsed_seconds=("elapsed_seconds", "sum"),
        modeled_elapsed_seconds=("modeled_elapsed_seconds", "sum"),
        **{column: (column, "sum") for column in distance_columns},
    ).reset_index()
    denominator = totals["modeled_elapsed_seconds"].where(
        totals["excluded_transitions"].eq(0) & totals["modeled_elapsed_seconds"].gt(0)
    )
    for distance, speed in component_columns.items():
        totals[speed] = totals[distance] / denominator
    return totals

zonal_pace_df = summarize_pace(zonal_transitions_df, [
    "source_dataset", "game_id", "period", "possession_id",
    "possession_team_id", "zone_segment_id", "zone",
])
display(zonal_pace_df.head(10))


,source_dataset,game_id,period,possession_id,possession_team_id,zone_segment_id,zone,transitions,sequences,excluded_transitions,same_second_transitions,elapsed_seconds,modeled_elapsed_seconds,distance_ft,distance_ew_ft,distance_ns_ft,distance_n_ft,speed_total_ft_s,speed_ew_ft_s,speed_ns_ft_s,speed_n_ft_s
0,NWHL,3,1,1,1,1,NZ,2,1,0,0,3.0,3.0,29.249081,15.0,25.0,25.0,9.749694,5.0,8.333333,8.333333
1,NWHL,3,1,1,1,2,OZ,1,1,0,0,2.0,2.0,6.0,0.0,6.0,6.0,3.0,0.0,3.0,3.0
2,NWHL,3,1,2,1,1,OZ,1,1,0,0,1.0,1.0,11.18034,5.0,10.0,0.0,11.18034,5.0,10.0,0.0
3,NWHL,3,1,3,5,1,NZ,1,1,0,0,2.0,2.0,34.655447,25.0,24.0,24.0,17.327723,12.5,12.0,12.0
4,NWHL,3,1,3,5,2,OZ,4,1,0,2,5.0,5.5,120.482546,75.0,81.0,76.0,21.905918,13.636364,14.727273,13.818182
5,NWHL,3,1,4,1,1,DZ,3,1,0,2,3.0,3.0,57.654288,33.0,47.0,47.0,19.218096,11.0,15.666667,15.666667
6,NWHL,3,1,5,5,1,OZ,1,1,0,0,2.0,2.0,3.162278,1.0,3.0,3.0,1.581139,0.5,1.5,1.5
7,NWHL,3,1,6,1,1,DZ,3,1,0,1,4.0,4.5,95.155822,73.0,60.0,60.0,21.145738,16.222222,13.333333,13.333333
8,NWHL,3,1,7,5,1,NZ,1,1,0,0,2.0,2.0,12.165525,2.0,12.0,12.0,6.082763,1.0,6.0,6.0
9,NWHL,3,1,7,5,2,OZ,1,1,0,0,2.0,2.0,38.0,0.0,38.0,38.0,19.0,0.0,19.0,19.0


## 3.5. Aggregate pace

### Sequence pace

Pool distance and modeled duration, keeping final movements and recorded totals for comparison. Incomplete, zero-duration, or transition-free sequences have missing pace. Modeled sequence spans can differ from recorded spans; they are estimates, not additional measured time.

In [57]:
# Include sequences with no transitions so coverage remains visible.
sequence_context = (
    augmented_events_df.sort_values("event_id", kind="stable")
    .groupby("possession_id", sort=False)
    .agg(
        source_dataset=("source_dataset", "first"), game_id=("game_id", "first"),
        period=("period", "first"), possession_team_id=("possession_team_id", "first"),
        is_5v5=("is_5v5", "first"), events=("event_id", "size"),
    ).reset_index()
)
sequence_pace_df = sequence_context.merge(
    summarize_pace(aggregation_df, ["possession_id"]).drop(columns="sequences"),
    on="possession_id", how="left", validate="one_to_one",
)
for column in ["transitions", "excluded_transitions", "same_second_transitions"]:
    sequence_pace_df[column] = sequence_pace_df[column].fillna(0).astype("Int64")
sequence_pace_df["pace_status"] = pd.Series("eligible", index=sequence_pace_df.index, dtype="string")
sequence_pace_df.loc[sequence_pace_df["modeled_elapsed_seconds"].eq(0), "pace_status"] = "zero_duration"
sequence_pace_df.loc[sequence_pace_df["excluded_transitions"].gt(0), "pace_status"] = "invalid_transitions"
sequence_pace_df.loc[sequence_pace_df["transitions"].eq(0), "pace_status"] = "no_transitions"

# Opponent identity is based on the sequence owner, not a contextual event team.
game_lookup = games_df.set_index("game_id")
home_ids = sequence_pace_df["game_id"].map(game_lookup["home_team_id"])
away_ids = sequence_pace_df["game_id"].map(game_lookup["away_team_id"])
sequence_pace_df["opponent_team_id"] = away_ids.where(
    sequence_pace_df["possession_team_id"].eq(home_ids), home_ids
).astype("Int64")

display(sequence_pace_df.groupby(["source_dataset", "pace_status"]).size().unstack(fill_value=0))
display(sequence_pace_df.head(10))


pace_status,eligible,no_transitions,zero_duration
source_dataset,,,
NWHL,5697,742,776
Olympics 2022,2198,369,215
Womens,4861,609,579


,possession_id,source_dataset,game_id,period,possession_team_id,is_5v5,events,transitions,excluded_transitions,same_second_transitions,elapsed_seconds,modeled_elapsed_seconds,distance_ft,distance_ew_ft,distance_ns_ft,distance_n_ft,speed_total_ft_s,speed_ew_ft_s,speed_ns_ft_s,speed_n_ft_s,pace_status,opponent_team_id
0,1,NWHL,3,1,1,True,4,3,0,0,5.0,5.0,35.249081,15.0,31.0,31.0,7.049816,3.0,6.2,6.2,eligible,5
1,2,NWHL,3,1,1,True,2,1,0,0,1.0,1.0,11.18034,5.0,10.0,0.0,11.18034,5.0,10.0,0.0,eligible,5
2,3,NWHL,3,1,5,True,6,5,0,2,7.0,7.5,155.137993,100.0,105.0,100.0,20.685066,13.333333,14.0,13.333333,eligible,1
3,4,NWHL,3,1,1,True,4,3,0,2,3.0,3.0,57.654288,33.0,47.0,47.0,19.218096,11.0,15.666667,15.666667,eligible,5
4,5,NWHL,3,1,5,True,3,1,0,0,2.0,2.0,3.162278,1.0,3.0,3.0,1.581139,0.5,1.5,1.5,eligible,1
5,6,NWHL,3,1,1,True,4,3,0,1,4.0,4.5,95.155822,73.0,60.0,60.0,21.145738,16.222222,13.333333,13.333333,eligible,5
6,7,NWHL,3,1,5,True,3,2,0,0,4.0,4.0,50.165525,2.0,50.0,50.0,12.541381,0.5,12.5,12.5,eligible,1
7,8,NWHL,3,1,1,True,4,3,0,2,3.0,3.0,41.219544,2.0,41.0,41.0,13.739848,0.666667,13.666667,13.666667,eligible,5
8,9,NWHL,3,1,5,True,12,11,0,6,13.0,13.5,403.066225,246.0,290.0,119.0,29.856757,18.222222,21.481481,8.814815,eligible,1
9,10,NWHL,3,1,1,True,3,1,0,1,0.0,0.0,0.0,0.0,0.0,0.0,<NA>,<NA>,<NA>,<NA>,zero_duration,5


### Team and game pace

Use complete positive-modeled-duration 5v5 sequences. Pool distance and modeled time for attacking and defending summaries. Exclude invalid or zero-modeled-duration zone segments from zonal team summaries.

In [58]:
# Initial team comparisons use complete, positive-duration 5v5 sequences.
complete_5v5 = sequence_pace_df.loc[
    sequence_pace_df["is_5v5"] & sequence_pace_df["pace_status"].eq("eligible")
]
sequence_lookup = sequence_pace_df.set_index("possession_id")
team_intervals = aggregation_df.loc[
    aggregation_df["possession_id"].isin(complete_5v5["possession_id"])
].copy()

# Defending pace is the opponent's attacking movement while this team defends.
def assign_team_roles(intervals):
    attacking = intervals.assign(role="attacking", team_id=intervals["possession_team_id"])
    defending = intervals.assign(
        role="defending",
        team_id=intervals["possession_id"].map(sequence_lookup["opponent_team_id"]),
    )
    return pd.concat([attacking, defending], ignore_index=True)

team_game_pace_df = summarize_pace(assign_team_roles(team_intervals), [
    "source_dataset", "game_id", "team_id", "role",
])
team_pace_df = summarize_pace(assign_team_roles(team_intervals), [
    "source_dataset", "team_id", "role",
])

# A zero-modeled-duration zone segment cannot supply a zone-specific speed.
# Exclude it from zonal team summaries rather than add distance without time.
valid_segments = zonal_pace_df.loc[
    zonal_pace_df["possession_id"].isin(complete_5v5["possession_id"])
    & zonal_pace_df["excluded_transitions"].eq(0)
    & zonal_pace_df["modeled_elapsed_seconds"].gt(0),
    ["possession_id", "zone_segment_id"],
]
zone_intervals = zonal_transitions_df.merge(
    valid_segments, on=["possession_id", "zone_segment_id"],
    how="inner", validate="many_to_one",
)
team_zone_pace_df = summarize_pace(assign_team_roles(zone_intervals), [
    "source_dataset", "team_id", "role", "zone",
])
for table in [team_game_pace_df, team_pace_df, team_zone_pace_df]:
    table["team_name"] = table["team_id"].map(teams_df.set_index("team_id")["team_name"])

display(team_pace_df)
display(team_zone_pace_df.head(12))


,source_dataset,team_id,role,transitions,sequences,excluded_transitions,same_second_transitions,elapsed_seconds,modeled_elapsed_seconds,distance_ft,distance_ew_ft,distance_ns_ft,distance_n_ft,speed_total_ft_s,speed_ew_ft_s,speed_ns_ft_s,speed_n_ft_s,team_name
0,NWHL,1,attacking,4443,1036,0,1810,6349.0,6393.1,116400.447936,61333.0,82888.0,61496.0,18.2072,9.593624,12.965228,9.619121,Boston Pride
1,NWHL,5,attacking,2189,541,0,920,2934.0,2951.030303,52974.799135,28424.0,37326.0,28343.0,17.951289,9.63189,12.648464,9.604442,Minnesota Whitecaps
2,NWHL,3,attacking,2678,670,0,1142,3472.0,3500.833333,60459.171969,32264.0,42559.0,32689.0,17.269937,9.216091,12.15682,9.337491,Connecticut Whale
3,NWHL,2,attacking,3542,889,0,1512,4749.0,4783.833333,80418.660134,42401.0,57647.0,45313.0,16.810506,8.863394,12.050378,9.472111,Buffalo Beauts
4,NWHL,6,attacking,4019,931,0,1652,5831.0,5886.75,98850.509615,54161.0,68482.0,50459.0,16.792035,9.200493,11.633244,8.571623,Toronto Six
5,NWHL,4,attacking,1986,499,0,848,2621.0,2638.954545,46945.888794,25993.0,32455.0,23866.0,17.789578,9.849734,12.298431,9.043733,Metropolitan Riveters
6,Womens,12,attacking,3678,709,0,1597,5206.0,5239.133333,95075.518521,49370.0,67807.0,49401.0,18.147184,9.423314,12.942408,9.429231,Canada (O18)
7,Womens,17,attacking,815,226,0,341,1206.0,1212.0,22031.316547,11443.0,15864.0,12980.0,18.177654,9.441419,13.089109,10.709571,Olympic Athletes from Russia (O18)
8,Womens,15,attacking,447,111,0,191,650.0,653.833333,10724.41417,5759.0,7565.0,5932.0,16.402367,8.808055,11.570227,9.072648,Finland (O18)
9,Womens,19,attacking,1653,305,0,750,2142.0,2157.75,41758.541454,21871.0,29876.0,21503.0,19.352817,10.136021,13.845904,9.965473,United States (O18)


,source_dataset,team_id,role,zone,transitions,sequences,excluded_transitions,same_second_transitions,elapsed_seconds,modeled_elapsed_seconds,distance_ft,distance_ew_ft,distance_ns_ft,distance_n_ft,speed_total_ft_s,speed_ew_ft_s,speed_ns_ft_s,speed_n_ft_s,team_name
0,NWHL,1,attacking,NZ,1178,500,0,537,1600.0,1600.333333,33091.896533,14792.0,26010.0,22607.0,20.678127,9.243074,16.252864,14.126432,Boston Pride
1,NWHL,1,attacking,OZ,1877,493,0,675,2946.0,2943.6,51994.909877,26541.0,37766.0,27391.0,17.663714,9.01651,12.829868,9.305272,Boston Pride
2,NWHL,5,attacking,NZ,578,247,0,270,764.0,767.166667,16115.155245,7291.0,12634.0,10966.0,21.006068,9.503802,16.46839,14.294156,Minnesota Whitecaps
3,NWHL,5,attacking,OZ,737,215,0,270,1059.0,1055.5,18907.513324,9598.0,13717.0,10382.0,17.913324,9.093321,12.995737,9.836097,Minnesota Whitecaps
4,NWHL,1,attacking,DZ,1352,566,0,562,1803.0,1849.166667,31313.641526,20000.0,19112.0,11498.0,16.933921,10.815683,10.335466,6.217936,Boston Pride
5,NWHL,5,attacking,DZ,865,333,0,371,1111.0,1128.363636,17952.130565,11535.0,10975.0,6995.0,15.90988,10.222768,9.726474,6.199243,Minnesota Whitecaps
6,NWHL,3,attacking,NZ,668,297,0,292,907.0,896.333333,17324.186084,7464.0,13737.0,12046.0,19.327839,8.327259,15.325772,13.439197,Connecticut Whale
7,NWHL,3,attacking,DZ,1011,382,0,458,1223.0,1258.0,19844.087588,12775.0,12107.0,8634.0,15.774314,10.155008,9.624006,6.863275,Connecticut Whale
8,NWHL,2,attacking,DZ,1361,521,0,600,1777.0,1832.5,27264.999174,16883.0,17246.0,12043.0,14.878581,9.213097,9.411187,6.571896,Buffalo Beauts
9,NWHL,2,attacking,NZ,917,407,0,438,1198.0,1188.166667,23942.206677,10640.0,19025.0,17056.0,20.150546,8.954973,16.012063,14.354888,Buffalo Beauts


## 3.6. Calculate polygrid pace

Follow paper section 2.4: divide distance and modeled time equally among intersected 5 ? 5 ft cells, then calculate speed from pooled distance and time. Keep the final movements as agreed. No additional zone boundaries or timing adjustments.


### Rink grid

Use the previous implementation's 668-cell layout: a 40 ? 17 grid with three cells removed at each corner. This approximates rounded corners; the paper does not specify the exact cuts. Edge and corner touches count as intersections, including stationary positions on cell boundaries. Coordinates remain in the attacking team's perspective for both roles.


In [59]:
from scipy.ndimage import gaussian_filter

CELL_SIZE_FT = 5
GRID_ROWS, GRID_COLUMNS = 17, 40
GAUSSIAN_SIGMA = 0.5

corner_cuts = {(0, 0), (1, 0), (0, 1)}
removed_cells = {
    (GRID_COLUMNS - 1 - col if flip_x else col,
     GRID_ROWS - 1 - row if flip_y else row)
    for col, row in corner_cuts
    for flip_x in [False, True] for flip_y in [False, True]
}
polygrid_cells_df = pd.DataFrame([
    {"grid_column": col, "grid_row": row,
     "x_min_ft": col * CELL_SIZE_FT, "x_max_ft": (col + 1) * CELL_SIZE_FT,
     "y_min_ft": row * CELL_SIZE_FT, "y_max_ft": (row + 1) * CELL_SIZE_FT}
    for row in range(GRID_ROWS) for col in range(GRID_COLUMNS)
    if (col, row) not in removed_cells
])
polygrid_cells_df.insert(0, "cell_id", np.arange(len(polygrid_cells_df)))

# Slab intersection includes cells touched only at an edge or corner.
cell_lower = polygrid_cells_df[["x_min_ft", "y_min_ft"]].to_numpy(float)
cell_upper = polygrid_cells_df[["x_max_ft", "y_max_ft"]].to_numpy(float)

def intersected_cells(x0, y0, x1, y1):
    start = np.array([x0, y0], dtype=float)
    delta = np.array([x1 - x0, y1 - y0], dtype=float)
    enter = np.zeros(len(polygrid_cells_df))
    leave = np.ones(len(polygrid_cells_df))
    inside = np.ones(len(polygrid_cells_df), dtype=bool)
    for axis in range(2):
        if abs(delta[axis]) < 1e-12:
            inside &= (start[axis] >= cell_lower[:, axis] - 1e-10)
            inside &= (start[axis] <= cell_upper[:, axis] + 1e-10)
        else:
            first = (cell_lower[:, axis] - start[axis]) / delta[axis]
            last = (cell_upper[:, axis] - start[axis]) / delta[axis]
            enter = np.maximum(enter, np.minimum(first, last))
            leave = np.minimum(leave, np.maximum(first, last))
    return np.flatnonzero(inside & (enter <= leave + 1e-10)).tolist()

assert len(polygrid_cells_df) == 668


### Transition contributions for replay

Store one row per transition and intersected cell, with event and possession identifiers. Keep valid zero-duration movements and stationary intervals. Paths entirely inside removed corner cells remain unassigned and appear in the assessment. For replay, accumulate contributions in `end_event_id` order and recompute distance / modeled time; smoothing is applied after pooling.


In [60]:
polygrid_intervals = aggregation_df.loc[aggregation_df["exclusion_reason"].isna()].copy()
polygrid_intervals["cell_ids"] = [
    intersected_cells(row.start_x, row.start_y, row.end_x, row.end_y)
    for row in polygrid_intervals.itertuples(index=False)
]
polygrid_unassigned_df = polygrid_intervals.loc[
    polygrid_intervals["cell_ids"].str.len().eq(0)
].drop(columns="cell_ids")

covered = polygrid_intervals.loc[polygrid_intervals["cell_ids"].str.len().gt(0)].copy()
covered["share"] = 1 / covered["cell_ids"].str.len()
replay_columns = [
    "source_dataset", "game_id", "period", "possession_id", "possession_team_id",
    "start_event_id", "end_event_id", "cell_ids", "share",
    "elapsed_seconds", "modeled_elapsed_seconds", *distance_columns,
]
polygrid_contributions_df = covered[replay_columns].explode("cell_ids").rename(
    columns={"cell_ids": "cell_id"}
).reset_index(drop=True)
polygrid_contributions_df["cell_id"] = polygrid_contributions_df["cell_id"].astype(int)
# The same share is applied to every component and both timing measures.
for column in ["elapsed_seconds", "modeled_elapsed_seconds", *distance_columns]:
    polygrid_contributions_df[column] *= polygrid_contributions_df["share"]
polygrid_contributions_df = polygrid_contributions_df.sort_values(
    ["end_event_id", "cell_id"], kind="stable"
).reset_index(drop=True)


### Source and team grids

Use the same complete, positive-duration 5v5 sequences as section 3.5. Pool contributions once for the source baseline; team grids distinguish attacking and defending roles. Zero-time distance remains in pooled totals, but cells with no modeled exposure have missing speed.


In [61]:
polygrid_5v5 = polygrid_contributions_df.loc[
    polygrid_contributions_df["possession_id"].isin(complete_5v5["possession_id"])
].copy()

def summarize_polygrid(contributions, keys):
    totals = contributions.groupby(keys, sort=False).agg(
        transitions=("end_event_id", "nunique"),
        sequences=("possession_id", "nunique"),
        games=("game_id", "nunique"),
        elapsed_seconds=("elapsed_seconds", "sum"),
        modeled_elapsed_seconds=("modeled_elapsed_seconds", "sum"),
        **{column: (column, "sum") for column in distance_columns},
    ).reset_index()
    denominator = totals["modeled_elapsed_seconds"].where(totals["modeled_elapsed_seconds"].gt(0))
    for distance, speed in component_columns.items():
        totals[speed] = totals[distance] / denominator
    # Complete each group's grid; unobserved cells remain missing.
    groups = contributions[keys[:-1]].drop_duplicates()
    full_grid = groups.merge(polygrid_cells_df, how="cross")
    return full_grid.merge(totals, on=keys, how="left", validate="one_to_one")

polygrid_source_pace_df = summarize_polygrid(polygrid_5v5, ["source_dataset", "cell_id"])
polygrid_team_pace_df = summarize_polygrid(assign_team_roles(polygrid_5v5), [
    "source_dataset", "team_id", "role", "cell_id",
])
polygrid_team_pace_df["team_name"] = polygrid_team_pace_df["team_id"].map(
    teams_df.set_index("team_id")["team_name"]
)


### Smoothed and differential pace

Apply a 2D Gaussian kernel (? = 0.5 cells) to each speed grid before subtraction, as in the paper. Normalize over observed cells and preserve missing cells and corner cuts. Report raw and smoothed team-minus-source differences for all four components. The source baseline includes the team's own observations.


In [62]:
def smooth_polygrids(table, keys):
    result = table.copy()
    for speed in speed_columns:
        result[f"{speed}_smoothed"] = np.nan
    for _, group in result.groupby(keys, sort=False):
        rows = group["grid_row"].to_numpy(int)
        cols = group["grid_column"].to_numpy(int)
        for speed in speed_columns:
            values = group[speed].to_numpy(float)
            observed = np.isfinite(values)
            image = np.zeros((GRID_ROWS, GRID_COLUMNS))
            mask = np.zeros_like(image)
            image[rows[observed], cols[observed]] = values[observed]
            mask[rows[observed], cols[observed]] = 1
            numerator = gaussian_filter(image, sigma=GAUSSIAN_SIGMA, mode="constant")
            denominator = gaussian_filter(mask, sigma=GAUSSIAN_SIGMA, mode="constant")
            smooth = np.divide(numerator, denominator,
                               out=np.full_like(image, np.nan), where=denominator > 0)
            result.loc[group.index, f"{speed}_smoothed"] = np.where(
                observed, smooth[rows, cols], np.nan
            )
    return result

polygrid_source_pace_df = smooth_polygrids(polygrid_source_pace_df, ["source_dataset"])
polygrid_team_pace_df = smooth_polygrids(polygrid_team_pace_df, ["source_dataset", "team_id", "role"])
comparison_columns = [*speed_columns, *[f"{speed}_smoothed" for speed in speed_columns]]
polygrid_differential_df = polygrid_team_pace_df.merge(
    polygrid_source_pace_df[["source_dataset", "cell_id", *comparison_columns]].rename(
        columns={column: f"source_{column}" for column in comparison_columns}
    ), on=["source_dataset", "cell_id"], how="left", validate="many_to_one",
)
for speed in speed_columns:
    polygrid_differential_df[f"{speed}_difference"] = (
        polygrid_differential_df[speed] - polygrid_differential_df[f"source_{speed}"]
    )
    polygrid_differential_df[f"{speed}_difference_smoothed"] = (
        polygrid_differential_df[f"{speed}_smoothed"]
        - polygrid_differential_df[f"source_{speed}_smoothed"]
    )


### Allocation assessment

Check that covered paths conserve distance and both time measures. Report corner paths without coverage explicitly; they are excluded from spatial grids only.


In [63]:
allocation_columns = ["elapsed_seconds", "modeled_elapsed_seconds", *distance_columns]
allocated = polygrid_contributions_df.groupby("end_event_id")[allocation_columns].sum()
expected = covered.set_index("end_event_id")[allocation_columns].reindex(allocated.index)
assert np.allclose(allocated.to_numpy(float), expected.to_numpy(float))
assert np.allclose(polygrid_contributions_df.groupby("end_event_id")["share"].sum(), 1)
assert polygrid_contributions_df["cell_id"].isin(polygrid_cells_df["cell_id"]).all()
assert not polygrid_contributions_df.duplicated(["end_event_id", "cell_id"]).any()

# Hand-check cell touches, stationary paths, and removed corners.
assert len(intersected_cells(12, 12, 13, 13)) == 1
assert len(intersected_cells(12, 12, 17, 12)) == 2
assert len(intersected_cells(15, 12, 15, 12)) == 2
assert intersected_cells(1, 1, 2, 2) == []
for table in [polygrid_source_pace_df, polygrid_team_pace_df]:
    positive_exposure = table["modeled_elapsed_seconds"].gt(0)
    assert table.loc[~positive_exposure, speed_columns].isna().all().all()
    for speed in speed_columns:
        assert table[f"{speed}_smoothed"].isna().equals(table[speed].isna())

display(pd.DataFrame([{
    "grid_cells": len(polygrid_cells_df), "valid_transitions": len(polygrid_intervals),
    "covered_transitions": len(covered), "unassigned_transitions": len(polygrid_unassigned_df),
    "transition_cell_contributions": len(polygrid_contributions_df),
    "unassigned_distance_ft": polygrid_unassigned_df["distance_ft"].sum(),
    "unassigned_modeled_seconds": polygrid_unassigned_df["modeled_elapsed_seconds"].sum(),
}]))
display(polygrid_differential_df.head())
if not polygrid_unassigned_df.empty:
    display(polygrid_unassigned_df.head())


,grid_cells,valid_transitions,covered_transitions,unassigned_transitions,transition_cell_contributions,unassigned_distance_ft,unassigned_modeled_seconds
0,668,63298,63298,0,475457,0.0,0.0


,source_dataset,team_id,role,cell_id,grid_column,grid_row,x_min_ft,x_max_ft,y_min_ft,y_max_ft,transitions,sequences,games,elapsed_seconds,modeled_elapsed_seconds,distance_ft,distance_ew_ft,distance_ns_ft,distance_n_ft,speed_total_ft_s,speed_ew_ft_s,speed_ns_ft_s,speed_n_ft_s,team_name,speed_total_ft_s_smoothed,speed_ew_ft_s_smoothed,speed_ns_ft_s_smoothed,speed_n_ft_s_smoothed,source_speed_total_ft_s,source_speed_ew_ft_s,source_speed_ns_ft_s,source_speed_n_ft_s,source_speed_total_ft_s_smoothed,source_speed_ew_ft_s_smoothed,source_speed_ns_ft_s_smoothed,source_speed_n_ft_s_smoothed,speed_total_ft_s_difference,speed_total_ft_s_difference_smoothed,speed_ew_ft_s_difference,speed_ew_ft_s_difference_smoothed,speed_ns_ft_s_difference,speed_ns_ft_s_difference_smoothed,speed_n_ft_s_difference,speed_n_ft_s_difference_smoothed
0,NWHL,1,attacking,0,2,0,10,15,0,5,7.0,3.0,3.0,2.995699,3.210192,16.288442,8.300514,12.358719,7.794203,5.073978,2.585676,3.849838,2.427956,Boston Pride,5.606248,2.915925,4.136685,2.793513,6.228124,3.086443,4.637225,3.058364,6.970596,3.607972,5.059049,3.449601,-1.154146,-1.364348,-0.500768,-0.692048,-0.787387,-0.922364,-0.630408,-0.656088
1,NWHL,1,attacking,1,3,0,15,20,0,5,25.0,16.0,7.0,10.261572,9.92498,66.180693,35.037777,43.753374,35.734951,6.668093,3.530262,4.408409,3.600506,Boston Pride,6.914663,3.530488,4.769964,3.826395,9.137243,4.812651,6.439402,4.854662,9.108826,4.760118,6.499944,4.872292,-2.46915,-2.194163,-1.282389,-1.229630,-2.030993,-1.729979,-1.254156,-1.045896
2,NWHL,1,attacking,2,4,0,20,25,0,5,35.0,21.0,7.0,12.016667,11.390575,94.399206,35.680754,74.621627,65.313294,8.287484,3.13248,6.551173,5.733977,Boston Pride,8.969908,3.569682,7.030598,6.073824,8.552043,3.832978,6.631091,5.369022,9.445457,4.317844,7.281258,5.849306,-0.26456,-0.475549,-0.700498,-0.748162,-0.079919,-0.250659,0.364955,0.224517
3,NWHL,1,attacking,3,5,0,25,30,0,5,61.0,38.0,7.0,17.269048,16.695368,174.660909,60.384687,142.784128,124.342461,10.461639,3.616853,8.55232,7.447722,Boston Pride,11.826392,4.257778,9.650902,8.509636,10.785722,4.097181,8.813212,7.560864,11.267888,4.471927,9.121047,7.724535,-0.324083,0.558503,-0.480329,-0.214149,-0.260892,0.529855,-0.113142,0.785101
4,NWHL,1,attacking,4,6,0,30,35,0,5,58.0,48.0,7.0,10.447078,10.92294,183.840194,55.973766,158.927124,153.411215,16.830651,5.124423,14.549848,14.044864,Boston Pride,16.807552,5.364986,14.464664,13.570809,11.647381,3.884854,9.935095,8.666806,12.438357,4.316124,10.559300,9.188672,5.183269,4.369195,1.239569,1.048862,4.614754,3.905364,5.378058,4.382137


## 3.7. Calculate individual player pace

Follow paper section 2.6.1: split each transition's distance and time equally between its endpoint players; allocate everything to one player when both endpoints name that player. Use the eligible 5v5 zone intervals from section 3.5 and their destination zones. Keep our modeled timing and final movements.

These are raw involvement-based pace estimates. Without shifts, lineups, and player positions, we cannot reproduce WOWY, the 200-minute ice-time threshold, or position-adjusted comparisons.


### Player contributions

Use each endpoint's primary `player_id`, not secondary players. Missing endpoint players retain an unattributed half-share. Store one row per transition and player, including unattributed shares, for traceability.


In [64]:
players_df = pd.read_parquet(PROCESSED_DATA_DIR / "players.parquet")
event_players = augmented_events_df.set_index("event_id")["player_id"]
player_allocation_columns = ["elapsed_seconds", "modeled_elapsed_seconds", *distance_columns]
player_interval_keys = [
    "source_dataset", "game_id", "period", "possession_id", "possession_team_id",
    "zone", "start_event_id", "end_event_id",
]

# Allocate half to each endpoint, then collapse identical endpoint players.
endpoint_shares = []
for endpoint in ["start_event_id", "end_event_id"]:
    shares = zone_intervals[player_interval_keys + player_allocation_columns].copy()
    shares["player_id"] = shares[endpoint].map(event_players).astype("Int64")
    shares["share"] = 0.5
    shares[player_allocation_columns] = shares[player_allocation_columns] * 0.5
    endpoint_shares.append(shares)

player_contributions_df = (
    pd.concat(endpoint_shares, ignore_index=True)
    .groupby(player_interval_keys + ["player_id"], dropna=False, sort=False)[
        ["share", *player_allocation_columns]
    ].sum().reset_index()
    .rename(columns={"possession_team_id": "team_id"})
    .sort_values(["end_event_id", "player_id"], kind="stable")
    .reset_index(drop=True)
)
player_contributions_df["player_id"] = player_contributions_df["player_id"].astype("Int64")


### Player pace by zone

Pool allocated distance and modeled time for each source, team, player, and zone. Calculate total, east-west, north-south, and north-only pace. Keep speeds missing when allocated modeled time is zero. Counts and allocated seconds describe involvement coverage, not time on ice.


In [65]:
attributed = player_contributions_df.loc[player_contributions_df["player_id"].notna()]
player_zone_pace_df = attributed.groupby(
    ["source_dataset", "team_id", "player_id", "zone"], sort=False
).agg(
    transitions=("end_event_id", "nunique"),
    sequences=("possession_id", "nunique"),
    games=("game_id", "nunique"),
    **{column: (column, "sum") for column in player_allocation_columns},
).reset_index()

player_duration = player_zone_pace_df["modeled_elapsed_seconds"].where(
    player_zone_pace_df["modeled_elapsed_seconds"].gt(0)
)
for distance, speed in component_columns.items():
    player_zone_pace_df[speed] = player_zone_pace_df[distance] / player_duration
player_zone_pace_df["player_name"] = player_zone_pace_df["player_id"].map(
    players_df.set_index("player_id")["player_name"]
)
player_zone_pace_df["team_name"] = player_zone_pace_df["team_id"].map(
    teams_df.set_index("team_id")["team_name"]
)
display(player_zone_pace_df.head(12))


,source_dataset,team_id,player_id,zone,transitions,sequences,games,elapsed_seconds,modeled_elapsed_seconds,distance_ft,distance_ew_ft,distance_ns_ft,distance_n_ft,speed_total_ft_s,speed_ew_ft_s,speed_ns_ft_s,speed_n_ft_s,player_name,team_name
0,NWHL,1,5,NZ,68,44,6,46.0,49.166667,1137.634699,468.5,927.5,784.5,23.138333,9.528814,18.864407,15.955932,Jillian Dempsey,Boston Pride
1,NWHL,1,12,NZ,127,66,7,139.5,136.75,2578.837672,1147.5,2089.5,1719.0,18.858045,8.391225,15.279707,12.570384,McKenna Brand,Boston Pride
2,NWHL,1,12,OZ,237,90,7,279.5,278.466667,5259.58438,2611.0,3933.5,2840.5,18.887662,9.376347,14.125569,10.200503,McKenna Brand,Boston Pride
3,NWHL,1,16,OZ,246,91,7,295.0,294.625,4943.88589,2690.5,3511.0,2320.0,16.780266,9.131947,11.916843,7.874417,Samantha Davis,Boston Pride
4,NWHL,1,19,OZ,249,106,7,352.5,352.583333,5634.652784,2602.0,4314.5,3211.5,15.981053,7.379816,12.236823,9.108485,Tereza Vanisova,Boston Pride
5,NWHL,5,104,NZ,58,31,4,50.0,51.166667,1022.581448,480.0,756.5,598.0,19.985305,9.381107,14.785016,11.687296,Stephanie Anderson,Minnesota Whitecaps
6,NWHL,5,104,OZ,108,43,4,92.0,91.666667,1584.377712,994.5,982.0,721.0,17.28412,10.849091,10.712727,7.865455,Stephanie Anderson,Minnesota Whitecaps
7,NWHL,5,100,OZ,96,44,4,97.0,95.5,1497.936108,785.0,1105.0,796.0,15.685195,8.219895,11.570681,8.335079,Meghan Lorence,Minnesota Whitecaps
8,NWHL,5,94,OZ,128,56,4,111.5,114.916667,2160.044405,1038.0,1588.5,1165.5,18.796616,9.032632,13.82306,10.142132,Haley Mack,Minnesota Whitecaps
9,NWHL,1,10,DZ,201,107,7,197.0,204.75,3456.081421,2380.0,1979.5,1153.0,16.879519,11.623932,9.667888,5.631258,Mallory Souliotis,Boston Pride


### Allocation and coverage assessment

Verify that attributed and unattributed shares conserve every transition's distance and time. Report unattributed exposure separately rather than transferring it to the known player.


In [66]:
player_allocated = player_contributions_df.groupby("end_event_id")[player_allocation_columns].sum()
player_expected = zone_intervals.set_index("end_event_id")[player_allocation_columns].reindex(
    player_allocated.index
)
assert len(player_allocated) == len(zone_intervals)
assert np.allclose(player_allocated.to_numpy(float), player_expected.to_numpy(float))
assert np.allclose(player_contributions_df.groupby("end_event_id")["share"].sum(), 1)
assert not player_contributions_df.duplicated(["end_event_id", "player_id"]).any()

# Same-player intervals receive one full share, including zero-time intervals.
start_players = zone_intervals["start_event_id"].map(event_players)
end_players = zone_intervals["end_event_id"].map(event_players)
same_player_ids = zone_intervals.loc[
    start_players.eq(end_players).fillna(False), "end_event_id"
]
same_player_rows = player_contributions_df.loc[
    player_contributions_df["end_event_id"].isin(same_player_ids)
]
assert len(same_player_rows) == len(same_player_ids)
assert same_player_rows["share"].eq(1).all()

player_lookup = players_df.set_index("player_id")
assert attributed["player_id"].isin(player_lookup.index).all()
assert attributed["player_id"].map(player_lookup["team_id"]).eq(attributed["team_id"]).all()
assert attributed["player_id"].map(player_lookup["source_dataset"]).eq(attributed["source_dataset"]).all()
assert player_zone_pace_df.loc[
    ~player_zone_pace_df["modeled_elapsed_seconds"].gt(0), speed_columns
].isna().all().all()
player_totals = player_zone_pace_df.groupby("source_dataset")[player_allocation_columns].sum()
attributed_totals = attributed.groupby("source_dataset")[player_allocation_columns].sum()
assert np.allclose(player_totals.sort_index().to_numpy(float), attributed_totals.sort_index().to_numpy(float))

player_allocation_coverage_df = player_contributions_df.assign(
    unattributed=lambda df: df["player_id"].isna(),
    unattributed_share=lambda df: df["share"].where(df["unattributed"], 0),
    unattributed_modeled_seconds=lambda df: df["modeled_elapsed_seconds"].where(df["unattributed"], 0),
    unattributed_distance_ft=lambda df: df["distance_ft"].where(df["unattributed"], 0),
).groupby("source_dataset").agg(
    transitions=("end_event_id", "nunique"),
    players=("player_id", "nunique"),
    modeled_elapsed_seconds=("modeled_elapsed_seconds", "sum"),
    unattributed_transition_shares=("unattributed_share", "sum"),
    unattributed_modeled_seconds=("unattributed_modeled_seconds", "sum"),
    unattributed_distance_ft=("unattributed_distance_ft", "sum"),
).reset_index()
display(player_allocation_coverage_df)


,source_dataset,transitions,players,modeled_elapsed_seconds,unattributed_transition_shares,unattributed_modeled_seconds,unattributed_distance_ft
0,NWHL,18739,126,26154.501515,0.0,0.0,0.0
1,Olympics 2022,8992,106,12331.666667,0.0,0.0,0.0
2,Womens,17391,199,24216.766667,0.0,0.0,0.0


## 3.8. Assess pace estimates

### Coverage and consistency

Assess exported endpoint clocks, interval coverage, and conservation of recorded time, modeled time, and distance across aggregates. Timing is supplied by notebook 02; pace never crosses possession boundaries.

In [67]:
# Verify endpoint identity, shared timing, and possession boundaries.
endpoint_events = augmented_events_df.set_index("event_id")
assert not pace_df.duplicated(["start_event_id", "end_event_id"]).any()
assert pace_df["end_event_id"].gt(pace_df["start_event_id"]).all()
for endpoint in ["start", "end"]:
    event_ids = pace_df[f"{endpoint}_event_id"]
    assert event_ids.isin(endpoint_events.index).all()
    for column in ["source_dataset", "game_id", "period", "possession_id", "possession_team_id"]:
        assert event_ids.map(endpoint_events[column]).eq(pace_df[column]).all()
    for clock in ["clock_seconds", "modeled_clock_seconds"]:
        assert np.allclose(
            event_ids.map(endpoint_events[clock]).to_numpy(float),
            pace_df[f"{endpoint}_{clock}"].to_numpy(float), equal_nan=True,
        )

# Zoning partitions each interval exactly once; all aggregates conserve inputs.
assert len(zonal_transitions_df) == len(aggregation_df)
assert not zonal_transitions_df["end_event_id"].duplicated().any()
conservation_columns = ["elapsed_seconds", "modeled_elapsed_seconds", *distance_columns]
valid_intervals = aggregation_df.loc[aggregation_df["exclusion_reason"].isna()]
for summary in [sequence_pace_df, zonal_pace_df]:
    expected = valid_intervals.groupby("source_dataset")[conservation_columns].sum()
    actual = summary.groupby("source_dataset")[conservation_columns].sum()
    assert np.allclose(actual.reindex(expected.index).to_numpy(float), expected.to_numpy(float))
for role in ["attacking", "defending"]:
    for summary, inputs in [(team_pace_df, team_intervals), (team_zone_pace_df, zone_intervals)]:
        expected = inputs.groupby("source_dataset")[conservation_columns].sum()
        actual = summary.loc[summary["role"].eq(role)].groupby("source_dataset")[conservation_columns].sum()
        assert np.allclose(actual.reindex(expected.index).to_numpy(float), expected.to_numpy(float))

pace_coverage_df = pace_df.assign(
    eligible=pace_df["exclusion_reason"].isna(),
    excluded=pace_df["exclusion_reason"].notna(),
    recorded_zero=pace_df["elapsed_seconds"].eq(0),
    modeled_zero=pace_df["modeled_elapsed_seconds"].eq(0),
    zero_time_movement=(pace_df["exclusion_reason"].isna()
                        & pace_df["modeled_elapsed_seconds"].eq(0) & pace_df["distance_ft"].gt(0)),
).groupby("source_dataset").agg(
    transitions=("end_event_id", "size"), eligible=("eligible", "sum"),
    excluded=("excluded", "sum"), recorded_zero=("recorded_zero", "sum"),
    modeled_zero=("modeled_zero", "sum"), zero_time_movement=("zero_time_movement", "sum"),
).reset_index()
display(pace_coverage_df)
display(sequence_pace_df.groupby(["source_dataset", "pace_status"]).size().unstack(fill_value=0))
print("Endpoint, possession, timing, and aggregate conservation checks passed.")


,source_dataset,transitions,eligible,excluded,recorded_zero,modeled_zero,zero_time_movement
0,NWHL,26777,26777,0,11711,3778,0
1,Olympics 2022,11258,11258,0,5136,1108,0
2,Womens,25263,25263,0,11394,3313,2


pace_status,eligible,no_transitions,zero_duration
source_dataset,,,
NWHL,5697,742,776
Olympics 2022,2198,369,215
Womens,4861,609,579


Endpoint, possession, timing, and aggregate conservation checks passed.


### Timing sensitivity and unusual speeds

Compare pooled pace using recorded and modeled durations on identical eligible 5v5 sequences with positive recorded duration. Keep tied-event distances in both calculations. Inspect speed quantiles and the fastest transitions without applying a cutoff. Fractional times are estimates, and extreme speeds require context.


In [68]:
# Compare identical 5v5 sequence samples and distances under both clocks.
# Recorded-zero movements remain in aggregate distance, as in modeled summaries.
sensitivity_sample = complete_5v5.loc[complete_5v5["elapsed_seconds"].gt(0)]
sensitivity_totals = sensitivity_sample.groupby("source_dataset").agg(
    sequences=("possession_id", "size"),
    recorded_seconds=("elapsed_seconds", "sum"),
    modeled_seconds=("modeled_elapsed_seconds", "sum"),
    **{column: (column, "sum") for column in distance_columns},
).reset_index()
sensitivity_rows = []
for distance, speed in component_columns.items():
    comparison = sensitivity_totals[["source_dataset", "sequences", "recorded_seconds", "modeled_seconds"]].copy()
    comparison["component"] = speed
    comparison["recorded_pace_ft_s"] = sensitivity_totals[distance] / comparison["recorded_seconds"]
    comparison["modeled_pace_ft_s"] = sensitivity_totals[distance] / comparison["modeled_seconds"]
    comparison["difference_ft_s"] = comparison["modeled_pace_ft_s"] - comparison["recorded_pace_ft_s"]
    comparison["difference_pct"] = 100 * comparison["difference_ft_s"] / comparison["recorded_pace_ft_s"].replace(0, np.nan)
    sensitivity_rows.append(comparison)
pace_timing_sensitivity_df = pd.concat(sensitivity_rows, ignore_index=True)
display(pace_timing_sensitivity_df)
print("Sequences with zero recorded duration, omitted only from this paired comparison:")
display(complete_5v5.loc[complete_5v5["elapsed_seconds"].eq(0)].groupby("source_dataset").size())

# Inspect extremes without imposing an arbitrary cutoff or deleting observations.
measured_speeds = pace_df.loc[can_calculate_speed]
display(measured_speeds.groupby("source_dataset")[speed_columns].quantile([0.5, 0.95, 0.99, 1]))
pace_extremes_df = measured_speeds.nlargest(20, "speed_total_ft_s")[[
    "source_dataset", "game_id", "period", "possession_id", "possession_team_id",
    "start_event_id", "end_event_id", "start_clock_seconds", "end_clock_seconds",
    "elapsed_seconds", "modeled_elapsed_seconds", "start_x", "start_y", "end_x", "end_y",
    "distance_ft", *speed_columns,
]].copy()
for endpoint in ["start", "end"]:
    pace_extremes_df[f"{endpoint}_event"] = pace_extremes_df[f"{endpoint}_event_id"].map(endpoint_events["event"])
display(pace_extremes_df)


,source_dataset,sequences,recorded_seconds,modeled_seconds,component,recorded_pace_ft_s,modeled_pace_ft_s,difference_ft_s,difference_pct
0,NWHL,4355,25956.0,26048.674242,speed_total_ft_s,17.406666,17.344738,-0.061928,-0.355773
1,Olympics 2022,1735,12074.0,12277.0,speed_total_ft_s,18.350224,18.046803,-0.303421,-1.653498
2,Womens,3635,24058.0,24146.433333,speed_total_ft_s,18.086104,18.019866,-0.066238,-0.366238
3,NWHL,4355,25956.0,26048.674242,speed_ew_ft_s,9.336107,9.302892,-0.033215,-0.355773
4,Olympics 2022,1735,12074.0,12277.0,speed_ew_ft_s,10.160924,9.992914,-0.168011,-1.653498
5,Womens,3635,24058.0,24146.433333,speed_ew_ft_s,9.681146,9.64569,-0.035456,-0.366238
6,NWHL,4355,25956.0,26048.674242,speed_ns_ft_s,12.270265,12.226611,-0.043654,-0.355773
7,Olympics 2022,1735,12074.0,12277.0,speed_ns_ft_s,12.749379,12.538568,-0.210811,-1.653498
8,Womens,3635,24058.0,24146.433333,speed_ns_ft_s,12.736595,12.689949,-0.046646,-0.366238
9,NWHL,4355,25956.0,26048.674242,speed_n_ft_s,9.235167,9.202311,-0.032856,-0.355773


Sequences with zero recorded duration, omitted only from this paired comparison:


source_dataset
NWHL             211
Olympics 2022    104
Womens           140
dtype: int64

speed_total_ft_s  speed_ew_ft_s  speed_ns_ft_s  \
source_dataset                                                        
NWHL           0.50          15.63314           6.75           10.0   
               0.95        114.017543           86.0           86.0   
               0.99        161.622518          132.0          132.0   
               1.00        528.458135          245.0          528.0   
Olympics 2022  0.50         16.994301       7.569231      10.666667   
               0.95        117.528651           90.0           90.0   
               0.99        159.518149          132.0          128.0   
               1.00        322.614321          304.0          272.0   
Womens         0.50         17.066048       7.333333           11.0   
               0.95        113.475983           88.0           84.0   
               0.99        157.796104          134.0          126.0   
               1.00        357.453493          336.0          357.0   

                     speed_n_ft_s  
source_dataset                     
NWHL           0.50      3.428571  
               0.95          74.0  
               0.99         126.0  
               1.00         528.0  
Olympics 2022  0.50           3.0  
               0.95          76.0  
               0.99         126.0  
               1.00         272.0  
Womens         0.50           3.4  
               0.95          74.0  
               0.99         122.0  
               1.00         357.0

,source_dataset,game_id,period,possession_id,possession_team_id,start_event_id,end_event_id,start_clock_seconds,end_clock_seconds,elapsed_seconds,modeled_elapsed_seconds,start_x,start_y,end_x,end_y,distance_ft,speed_total_ft_s,speed_ew_ft_s,speed_ns_ft_s,speed_n_ft_s,start_event,end_event
13611,NWHL,7,2,3704,4,19212,19213,251.0,251.0,0.0,0.090909,49.0,11.0,97.0,9.0,48.041649,528.458135,22.0,528.0,528.0,Puck Recovery,Pass
13615,NWHL,7,2,3705,5,19217,19218,251.0,251.0,0.0,0.090909,11.0,72.0,47.0,78.0,36.496575,401.462327,66.0,396.0,396.0,Pass,Pass Reception
18250,NWHL,11,1,4920,1,25620,25621,674.0,674.0,0.0,0.333333,30.0,0.0,154.0,0.0,124.0,372.0,0.0,372.0,372.0,Pass,Pass Reception
35978,Womens,26,3,9367,19,50007,50008,320.0,320.0,0.0,0.333333,72.0,80.0,191.0,74.0,119.151164,357.453493,18.0,357.0,357.0,Pass,Pass Reception
29142,Womens,23,1,7814,12,40824,40825,86.0,86.0,0.0,0.142857,62.0,53.0,78.0,5.0,50.596443,354.175098,336.0,112.0,112.0,Pass,Pass Reception
7112,NWHL,5,1,2000,4,10110,10111,969.0,969.0,0.0,0.25,7.0,69.0,94.0,53.0,88.45903,353.83612,64.0,348.0,348.0,Pass,Pass Reception
19381,NWHL,11,3,5236,1,27199,27200,673.0,673.0,0.0,0.333333,79.0,32.0,194.0,58.0,117.902502,353.707506,78.0,345.0,345.0,Pass,Pass Reception
7098,NWHL,5,1,1994,4,10088,10089,995.0,995.0,0.0,0.166667,56.0,26.0,110.0,6.0,57.58472,345.508321,120.0,324.0,324.0,Pass,Pass Reception
856,NWHL,3,2,231,1,1188,1189,505.0,505.0,0.0,0.5,32.0,67.0,195.0,28.0,167.600716,335.201432,78.0,326.0,326.0,Pass,Pass Reception
6702,NWHL,4,3,1895,2,9525,9526,570.0,570.0,0.0,0.333333,9.0,36.0,106.0,84.0,108.226614,324.679842,144.0,291.0,291.0,Pass,Pass Reception


## 3.9. Export pace files

Export the tables below to `data/processed` as Snappy-compressed Parquet, preserving floating-point timing and identifiers. Replay contributions cover valid intervals; source and team grids use eligible 5v5 sequences. Player tables use eligible 5v5 zone intervals. Save a manifest documenting table grains and methodological choices, and verify every file by reading it back.


In [69]:
import json

pace_exports = {
    "pace_transitions": (pace_df, "Transition within possession; eligibility, recorded and modeled timing, speeds"),
    "pace_zonal_transitions": (zonal_transitions_df, "Transition with destination zone and zone segment"),
    "pace_zonal_segments": (zonal_pace_df, "Possession x zone segment"),
    "pace_sequences": (sequence_pace_df, "Possession, including sequences without calculable pace"),
    "pace_team_games": (team_game_pace_df, "Source x game x team x role; eligible 5v5"),
    "pace_teams": (team_pace_df, "Source x team x role; eligible 5v5"),
    "pace_team_zones": (team_zone_pace_df, "Source x team x role x zone; eligible 5v5 zone segments"),
    "pace_polygrid_cells": (polygrid_cells_df, "Cell geometry; 668 approximate rink cells"),
    "pace_polygrid_contributions": (polygrid_contributions_df, "Valid transition x cell; allocated distance and time; all strengths"),
    "pace_polygrid_sources": (polygrid_source_pace_df, "Source x cell; eligible 5v5"),
    "pace_polygrid_teams": (polygrid_team_pace_df, "Source x team x role x cell; eligible 5v5"),
    "pace_polygrid_differentials": (polygrid_differential_df, "Source x team x role x cell; raw and smoothed team-minus-source pace"),
    "pace_polygrid_unassigned": (polygrid_unassigned_df, "Valid transition without grid coverage; all strengths"),
    "pace_player_contributions": (player_contributions_df, "Eligible 5v5 zone transition x player share, including unattributed shares"),
    "pace_player_zones": (player_zone_pace_df, "Source x team x player x zone; individual involvement pace"),
}
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
pace_export_rows = []
for name, (table, grain) in pace_exports.items():
    export_path = PROCESSED_DATA_DIR / f"{name}.parquet"
    table.to_parquet(export_path, engine="pyarrow", compression="snappy", index=False)
    restored = pd.read_parquet(export_path)
    pd.testing.assert_frame_equal(table.reset_index(drop=True), restored)
    pace_export_rows.append({
        "file": export_path.name, "rows": len(table), "columns": len(table.columns),
        "grain": grain, "bytes": export_path.stat().st_size,
    })
pace_export_summary_df = pd.DataFrame(pace_export_rows)
pace_manifest = {
    "producer": "notebooks/03_pace_components.ipynb",
    "inputs": ["events_augmented.parquet", "event_transitions.parquet", "teams.parquet", "games.parquet", "players.parquet"],
    "units": {"distance": "feet", "time": "seconds", "speed": "feet/second"},
    "timing": "Recorded clocks preserved; speeds use notebook 02 modeled clocks. Fractions are estimated, not measured.",
    "sequences": "Movement stays within possession; final movements retained.",
    "zones": "Entire transition assigned to destination zone.",
    "polygrid": {"cell_size_ft": CELL_SIZE_FT, "cells": len(polygrid_cells_df),
                 "corners": "Three removed cells per corner; approximate layout",
                 "allocation": "Equal distance/time share across all touched cells, including edge and corner touches",
                 "smoothing_sigma_cells": GAUSSIAN_SIGMA,
                 "smoothing": "Normalized Gaussian speed smoothing before subtraction; missing cells preserved",
                 "perspective": "Attacking team for both attacking and defending roles",
                 "baseline": "Source pool includes each eligible 5v5 transition once, including the team being compared"},
    "players": "Equal endpoint shares; identical players receive full share. Missing shares stay unattributed. Allocated time is not ice time; no WOWY or position adjustment.",
    "tables": pace_export_rows,
}
manifest_path = PROCESSED_DATA_DIR / "pace_export_manifest.json"
manifest_path.write_text(json.dumps(pace_manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
assert json.loads(manifest_path.read_text(encoding="utf-8")) == pace_manifest
display(pace_export_summary_df)
print(f"Verified {len(pace_exports)} Parquet files and {manifest_path.name} in {PROCESSED_DATA_DIR}.")


,file,rows,columns,grain,bytes
0,pace_transitions.parquet,63298,25,"Transition within possession; eligibility, rec...",2383940
1,pace_zonal_transitions.parquet,63298,31,Transition with destination zone and zone segment,2607472
2,pace_zonal_segments.parquet,21789,21,Possession x zone segment,720033
3,pace_sequences.parquet,16046,22,"Possession, including sequences without calcul...",566841
4,pace_team_games.parquet,136,19,Source x game x team x role; eligible 5v5,19072
5,pace_teams.parquet,40,18,Source x team x role; eligible 5v5,15189
6,pace_team_zones.parquet,120,19,Source x team x role x zone; eligible 5v5 zone...,22549
7,pace_polygrid_cells.parquet,668,7,Cell geometry; 668 approximate rink cells,9178
8,pace_polygrid_contributions.parquet,475457,15,Valid transition x cell; allocated distance an...,3479067
9,pace_polygrid_sources.parquet,2004,25,Source x cell; eligible 5v5,291150


Verified 15 Parquet files and pace_export_manifest.json in ..\data\processed.
